# Submission 2 — Reflection and the Lightweight Challenge — SOLUTION KEY
### ME 323 Module 1 (staff only)

Your beam has been printed and tested. Three jobs here:

1. **Recall** the module's ideas from memory (no peeking).
2. **Reflect**: how did your beam actually do vs what your model promised?
3. **The lightweight challenge**: design the **lightest** beam that will
   **confidently hold 700 N** in three-point bending. Not printed — graded on
   the rationale, and graded hard: a beam expected to fail under 700 N loses
   heavily, and so does one padded far beyond the target. *Using the
   uncertainty is the whole assignment.*

## 0. Recall *(write before you compute; corrections earn credit, bluffing does not)*

1. Name the three failure modes and how the governing one is chosen. Which
   region of the (b, H_web) box does each own?
2. Pre-lab 1 calibrated σ_y, k, and c_s. For each: was it a correction or a
   confession? (One sentence each.)
3. A GP returns μ and σ at every design. Which one drove explore-vs-exploit,
   and what does ψ trade off?
4. Both class beams measured below their predictions. Give one reason each.
5. Name the four modeling lanes from Submission 1 and the one-line idea of each.

## 1. Your beam's test result

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
CS = 1.0                          # web shear-strength multiplier    (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_g"] = mass_g(df.b, df.H)
print(len(df), "tested beams")

new = pd.DataFrame([dict(beam_id=15, b=1.25, H=13.4, strength_N=460.8),
                    dict(beam_id=16, b=3.2, H=14.14, strength_N=609.0)])
new["mass_g"] = mass_g(new.b, new.H)
df = pd.concat([df, new], ignore_index=True)
df["str_to_weight"] = df.strength_N / df.mass_g

# >>> ENTER your group's final design and its measured result:
b_mine, H_mine = None, None        # your Submission 1 design (mm)
P_mine = None                      # measured failure load (N)
note_mine = ""                     # what the failure looked like
if P_mine is not None:
    sw_mine = P_mine / mass_g(b_mine, H_mine)
    print(f"your beam: ({b_mine}, {H_mine})  {P_mine} N  ->  {sw_mine:.1f} N/g")
    print(f"class scoreboard: best tested so far {df.str_to_weight.max():.1f} N/g")

loaded from GitHub
14 tested beams


**Reflect (memo):** your model predicted a mean and a σ for this beam. Was the
measured value inside μ ± 2σ? If not — was the model wrong, the print
different, or the failure a mode outside the model?

## 2. The lightweight challenge: hold 700 N, weigh as little as possible

Same 16 beams, same tools — different objective. Now strength is a
**constraint**, not the prize. The class-default confidence rule: require the
model's *pessimistic* strength (mean minus 2σ, in log space) to clear the
target:

$$P_{lo}(b, H) = e^{\,\mu_{\ln P}(b,H) - 2\sigma(b,H)} \;\ge\; 700\text{ N}$$

Among designs that pass, take the lightest. **FILL IN** the two marked lines.
(You may argue a different z than 2 in your memo — that is a risk posture,
not a math fact.)

In [2]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    beta = lambda t, a: 1 - 0.63*(t/a) + 0.052*(t/a)**5
    J  = (1/3)*beta(b_, h_)*h_*b_**3 + (2/3)*beta(tf_, B_)*B_*tf_**3
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c, b=b_, h=h_)

def P_bend(p, sy):
    return 4*sy*p["Ix"] / (p["c"] * L/1e3)
def P_shear(p, sy, cs):
    return 2 * (cs*sy/np.sqrt(3)) * p["b"]*p["h"]
def P_vm(Pb, Ps):
    return 1.0/np.sqrt(1/Pb**2 + 1/Ps**2)
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)
def capacity(b, H, sy, k, cs):
    p = section_props(b, H)
    return min(P_vm(P_bend(p, sy), P_shear(p, sy, cs)), P_LTB(p, sy, k))
def gov_mode(b, H, sy, k, cs):
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_shear(p, sy, cs), P_LTB(p, sy, k)
    return "shear" if Ps < min(Pb, Pl) else ("LTB" if Pl < 0.999*Pb else "bend")

SY_CAL, K_CAL, CS_CAL = 6.650e+07, 0.377, 2.25
P_TARGET = 700.0

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF

# class-default model: lane A (log str/w), 3% noise — swap in your own lane if you prefer
X = df[["b", "H"]].values
fmu, fsd = X.mean(0), X.std(0) + 1e-12
y = np.log(df.str_to_weight.values)
ymean = y.mean()
gp = GaussianProcessRegressor(C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
                              alpha=0.03**2, normalize_y=False,
                              n_restarts_optimizer=5, random_state=0).fit((X-fmu)/fsd, y-ymean)

bg = np.linspace(1.25, 7.0, 60); Hg = np.linspace(5.0, 16.0, 60)
BB, HH = np.meshgrid(bg, Hg)
Xg = np.column_stack([BB.ravel(), HH.ravel()])
mu_c, std = gp.predict((Xg - fmu)/fsd, return_std=True)
mass_grid = mass_g(Xg[:, 0], Xg[:, 1])
# strength = str/w * mass, so in logs: ln P = (mu + ymean) + ln(mass)
mu_lnP = mu_c + ymean + np.log(mass_grid)

P_lo = np.exp(mu_lnP - 2*std)
feasible = P_lo >= P_TARGET
masked = np.where(feasible, mass_grid, np.inf)
i = int(np.argmin(masked))
b_lt, H_lt = float(Xg[i, 0]), float(Xg[i, 1])
print(f"LIGHTWEIGHT DESIGN: b = {b_lt:.2f} mm, H_web = {H_lt:.2f} mm")
print(f"  mass {mass_grid[i]:.1f} g,  P_lo {P_lo[i]:.0f} N,  "
      f"model mean {np.exp(mu_lnP[i]):.0f} N")
print(f"  calibrated-physics check: {capacity(b_lt, H_lt, SY_CAL, K_CAL, CS_CAL):.0f} N, "
      f"mode {gov_mode(b_lt, H_lt, SY_CAL, K_CAL, CS_CAL)}")
print("\nCHECKPOINT (class-default model): you should arrive at "
      "b = 4.47, H_web = 13.39, mass = 21.7 g.")
print("If you are not getting that, check your work or talk to a TA.")

LIGHTWEIGHT DESIGN: b = 4.47 mm, H_web = 13.39 mm
  mass 21.7 g,  P_lo 700 N,  model mean 734 N
  calibrated-physics check: 738 N, mode bend

CHECKPOINT (class-default model): you should arrive at b = 4.47, H_web = 13.39, mass = 21.7 g.
If you are not getting that, check your work or talk to a TA.


## Memo (graded)

1. **The margin.** Your design's model-mean strength is well above 700 N; its
   P_lo is just above. In newtons, how much of your mass budget is buying
   *uncertainty* rather than *strength*? (Compare against the infeasible
   design just below yours in mass.)
2. **The z.** Defend z = 2, or argue and price a different z. What failure
   probability are you implicitly accepting, and what does the module's data
   say about whether the model's σ can be trusted for that arithmetic?
3. **Physics veto.** Does the calibrated capacity agree your design clears
   700 N? If model and physics disagree, whose side are you on, and why?
4. **One more test.** If staff offered you a single extra print-and-test
   before committing, which (b, H_web) would you test — and would you pick it
   by μ, by σ, or by the constraint boundary?